# 03 · Filtering & ranking (Stage 5)

The decisive stage. (a) self-consistency, (b) **whole-reaction-coordinate
preorganization** scored by the *worst* mechanistic step, and a combined
diversity-aware ranking.

In [1]:
import sys, os
# make the package importable from the notebooks/ directory
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
EXAMPLES = os.path.join(ROOT, "examples")
print("project root:", ROOT)


project root: /home/user/biofx_python/enzyme_design


## Stage 5a — self-consistency hard filters

In [2]:
from enzyme_design.metrics import read_metrics_csv, Thresholds
rows = read_metrics_csv(os.path.join(EXAMPLES, 'af2_metrics.csv'))
for m in rows:
    ok, reasons = m.evaluate()
    print(f"{m.design_id}: {'PASS' if ok else 'FAIL'}", '' if ok else reasons)

design_0001: PASS 
design_0002: PASS 
design_0003: FAIL ['rmsd=2.6 fails <=2', 'motif_ca_rmsd=1.9 fails <=1.5', 'plddt=72 fails >=80', 'min_pae=7.8 fails <=5', 'ligand_rmsd=6.1 fails <=5', 'ptm=0.71 fails >=0.8', 'iptm=0.65 fails >=0.8']
design_0004: PASS 
design_0005: PASS 
design_0006: PASS 
design_0007: FAIL ['rmsd=3.1 fails <=2', 'motif_ca_rmsd=2.5 fails <=1.5', 'plddt=65 fails >=80', 'min_pae=9.2 fails <=5', 'ligand_rmsd=7.5 fails <=5', 'ptm=0.62 fails >=0.8', 'iptm=0.58 fails >=0.8']
design_0008: PASS 


## Stage 5b — preorganization across the reaction coordinate
We rank by the **worst step**, not the average. `design_0004` looks fine
on average but collapses at the *intermediate* — exactly the failure mode
the protocol says single-TS filtering would miss.

In [3]:
import json
from enzyme_design.preorg import preorg_from_rmsd_samples
data = json.load(open(os.path.join(EXAMPLES, 'preorg_ensembles.json')))
profiles = {d: preorg_from_rmsd_samples(d, v['rmsd_samples'], v.get('contacts'))
            for d, v in data['designs'].items()}
for did, p in profiles.items():
    w = p.worst_step()
    print(f'{did}: worst={w.state} (score {p.worst_score():.2f}), '
          f'mean {p.mean_score():.2f}, passes={p.passes()}, '
          f'failing={p.failing_steps()}')

design_0001: worst=ts2 (score 0.95), mean 0.71, passes=True, failing=[]
design_0002: worst=ts2 (score 1.39), mean 1.15, passes=True, failing=[]
design_0004: worst=intermediate (score 2.95), mean 1.30, passes=False, failing=['intermediate']
design_0006: worst=ts2 (score 0.74), mean 0.54, passes=True, failing=[]
design_0005: worst=ts2 (score 1.64), mean 1.40, passes=True, failing=[]
design_0008: worst=ts2 (score 1.15), mean 0.93, passes=True, failing=[]


## Combined ranking + diverse selection

In [4]:
from enzyme_design.selection import rank_designs, select_diverse
clusters = {'design_0001':'A','design_0006':'A','design_0004':'A',
            'design_0002':'B','design_0005':'B','design_0008':'B',
            'design_0003':'C','design_0007':'C'}
ranked = rank_designs(rows, profiles, clusters)
print(f"{'design':14}{'pass':6}{'score':8}{'cluster':8}")
for r in ranked:
    print(f'{r.design_id:14}{str(r.passes_filters):6}{r.score:<8.3f}{r.cluster}')
print()
panel = select_diverse(ranked, n=3, per_cluster=1)
print('selected panel (diverse):', [(p.design_id, p.cluster) for p in panel])

design        pass  score   cluster 
design_0006   True  0.010   A
design_0001   True  0.073   A
design_0008   True  0.203   B
design_0002   True  0.262   B
design_0005   True  0.379   B
design_0004   False 0.480   A
design_0003   False 0.736   C
design_0007   False 0.870   C

selected panel (diverse): [('design_0006', 'A'), ('design_0008', 'B'), ('design_0001', 'A')]
